# Network Biology & Drug Repurposing with Multiomics
### Google Colab • Human breast cancer / TNBC research workflow

**Start:** Upload this notebook at https://colab.research.google.com → File → Upload notebook. Use a CPU runtime and **Runtime → Run all**. Default `MODE="demo"` generates all inputs automatically; no accounts or API keys are needed for the demonstration. Internet is needed for package installation.

**Pipeline:** expression + mutation + optional proteomics → quality checks → protein network → modules and centrality → integrated gene evidence → pathway enrichment → drug–target mapping → degree-matched network proximity + signature reversal → ranked research candidates → Cytoscape and CSV exports.

**Scope:** demo measurements, edges, gene sets, compounds and signatures are synthetic, including intentionally implanted signals. They are not TCGA results or evidence for drug efficacy. Real mode starts from processed differential-expression results, not FASTQ files. Scores generate hypotheses; they do not establish treatment suitability. Transcriptomic downregulation does not imply a gene should be activated, and mutation frequency is not a functional direction-of-effect estimate.

| Screenshot resource | Implementation in this notebook |
|---|---|
| TCGA / GDC | Live TCGA-BRCA file discovery; import a cohort-specific differential-expression table |
| cBioPortal | Study/profile discovery and explicit-sample mutation fetch helper; mutation import |
| STRING | Identifier mapping and live network helper; full background network CSV import |
| NetworkX | Communities, centrality, shortest paths, degree-matched randomization |
| Enrichr | Retrieve GO, Reactome and KEGG gene-set libraries using GSEApy |
| Reactome / Gene Ontology / KEGG | Local enrichment with a measured-gene background and BH correction |
| g:Profiler | Optional live enrichment using a custom background |
| DGIdb / Open Targets | Standardized evidence CSV import with source records; no fabricated live API response |
| LINCS | Import prefiltered level-5 signature CSVs; context-specific reversal analysis |
| Cytoscape | GraphML, node and edge exports for desktop import |
| Colab | Installation, input upload and result download cells |

The red unlabeled logo in the screenshot cannot be identified confidently and is not assigned a package name.


In [ ]:
%pip -q install "numpy>=1.26,<3" "pandas>=2.2,<3" "scipy>=1.12,<2" "networkx>=3.3,<4" "matplotlib>=3.8,<4" "requests>=2.31,<3" "gseapy>=1.1,<2" "gprofiler-official>=1.0,<2"


## 1. Configuration
Leave demo mode on for the first run. Real mode requires the tables described below. Use a new `RUN_NAME` for each analysis. Live requests are disabled by default; enabling them transmits gene identifiers to public services. Exact installed versions and input checksums are recorded at export.


In [ ]:
from pathlib import Path
import json, hashlib, platform, importlib.metadata, shutil, io, time
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import requests
from scipy.stats import ttest_ind, spearmanr, hypergeom
from IPython.display import display

MODE = "demo"                       # "demo" or "real"
RUN_NAME = "multiomics_demo"         # change for a new analysis
INPUT = Path("multiomics_inputs") / MODE
OUT = Path("multiomics_results") / MODE / RUN_NAME
INPUT.mkdir(exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
N_PERM = 200                         # use >=1000 for final research analysis
FDR = 0.05
MIN_ABS_LOG2FC = 1.0
MIN_SIGNATURE_GENES = 20
LIVE_ENRICHMENT = False
STRING_SCORE = 0.7
assert MODE in {"demo", "real"}
rng = np.random.default_rng(SEED)

def bh(p):
    p = np.asarray(p, dtype=float)
    out = np.full(len(p), np.nan)
    ok = np.isfinite(p)
    v = p[ok]
    if not len(v): return out
    order = np.argsort(v)
    q = np.minimum.accumulate((v[order] * len(v) / np.arange(1, len(v)+1))[::-1])[::-1]
    back = np.empty(len(v)); back[order] = np.minimum(q, 1)
    out[ok] = back
    return out

def require(df, cols, name):
    missing = set(cols) - set(df.columns)
    if missing: raise ValueError(f"{name}: missing columns {sorted(missing)}")

def read_table(name, cols, optional=False):
    path = INPUT / name
    if optional and not path.exists(): return pd.DataFrame(columns=cols)
    if not path.exists(): raise FileNotFoundError(f"Upload {path}; see the input specification.")
    df = pd.read_csv(path)
    require(df, cols, name)
    return df

def numeric(df, cols, name):
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors="raise")
        if not np.isfinite(df[col]).all(): raise ValueError(f"{name}: nonfinite {col}")

def unique(df, cols, name):
    if df.duplicated(cols).any(): raise ValueError(f"{name}: duplicate key {cols}; resolve before analysis")

def rank01(s):
    # Preserve missing values; a constant observed channel has neutral rank.
    return s.rank(method="average", pct=True)


## 2. Input contracts and cohort definition
All gene identifiers must be **approved human HGNC symbols**, consistently mapped across tables. Resolve aliases and one-to-many mappings before upload; do not silently uppercase or collapse identifiers. Each real input must have a row in `provenance.csv` with `file,source,release,cohort,notes`.

| File | Required columns | Meaning |
|---|---|---|
| `expression_de.csv` | `gene,log2FC,pvalue,padj` | ALL tested genes, not only significant hits; disease minus control |
| `samples.csv` | `sample_id,group,mutation_profiled` | One independent tumor per patient for mutation analysis; group is `case` or `control`; profiled is 0/1 |
| `mutations.csv` | `sample_id,gene,variant_class` | Non-synonymous somatic calls; `variant_class` is `nonsynonymous`, `synonymous`, or `unknown` |
| `ppi_edges.csv` | `source,target,score` | Broad background interactome, score in [0,1], not a DEG-only subgraph |
| `drug_targets.csv` | `drug_id,drug_name,gene,action,source,reference` | Curated direct target evidence, e.g. DGIdb/Open Targets; stable drug IDs |
| `signatures.csv` | `signature_id,drug_id,gene,z,cell_line,dose,time` | Long-format drug-versus-vehicle level-5-like expression signature |
| `proteomics_de.csv` (optional) | `gene,log2FC,padj` | Processed protein disease-versus-control results |
| `open_targets.csv` (optional) | `gene,disease_id,association_score,source` | One preselected disease, evidence annotation only |
| `pathways.gmt` (optional) | GMT text | One row per set: name, description, then gene symbols, tab separated |
| `provenance.csv` | `file,source,release,cohort,notes` | Database releases, cohort selection, identifier mapping and processing |

For TCGA-BRCA/TNBC, define the tumor subset with clinical ER, PR and HER2 status and exclude ambiguous records; basal subtype is not interchangeable with TNBC. Match aliquots/patients across assays. Mutation denominators include **all mutation-profiled case samples**, including those with no variant in the selected genes. Do not count missing sequencing as wild type. Panel data require per-gene callable denominators; this notebook assumes a common assay covering the genes under analysis.

Produce real differential expression separately with a count-aware method such as DESeq2/edgeR, accounting for pairing, batches and covariates. Do not feed raw RNA counts into the demo Welch test. Optional proteomics is integrated at the summary-evidence level; this is not a latent-factor multiomics model.

For LINCS, choose relevant cell line, dose, duration and QC-passing signatures **before analysis**. Map LINCS gene/compound IDs to the same HGNC/drug IDs. Prefer measured landmark genes; document inferred genes if used. Upload a manageable subset rather than a full GCTX release. The reversal statistic here is negative Spearman correlation, **not CMap tau**.


In [ ]:
# Optional Colab upload: uncomment when MODE="real", then place inputs in INPUT.
# from google.colab import files
# for name, content in files.upload().items():
#     (INPUT / Path(name).name).write_bytes(content)


## 3. Create an entirely synthetic demonstration
Synthetic symbols and compound IDs prevent toy findings from being mistaken for actual biological evidence. Expression and protein matrices are simulated on an already normalized log2 scale. Mutation calls use a separate profiled-sample denominator.


In [ ]:
if MODE == "demo":
    genes = [f"DEMO_G{i:03d}" for i in range(120)]
    sample_ids = [f"DEMO_S{i:02d}" for i in range(24)]
    samples_demo = pd.DataFrame({"sample_id":sample_ids, "group":["control"]*12+["case"]*12,
                                 "mutation_profiled":[0]*12+[1]*12})
    effect = np.zeros(120); effect[:15] = 2; effect[15:30] = -2
    x = rng.normal(7, 0.7, (120,24)); x[:,12:] += effect[:,None]
    p = ttest_ind(x[:,12:], x[:,:12], axis=1, equal_var=False).pvalue
    demo_de = pd.DataFrame({"gene":genes,"log2FC":x[:,12:].mean(1)-x[:,:12].mean(1),
                            "pvalue":p,"padj":bh(p)})
    pd.DataFrame(x,index=pd.Index(genes,name="gene"),columns=sample_ids).to_csv(INPUT/"demo_log2_expression.csv")
    demo_de.to_csv(INPUT/"expression_de.csv",index=False)
    samples_demo.to_csv(INPUT/"samples.csv",index=False)
    muts = [(s,g,"nonsynonymous") for s in sample_ids[12:] for i,g in enumerate(genes)
            if rng.random() < (0.35 if i<20 else 0.02)]
    pd.DataFrame(muts,columns=["sample_id","gene","variant_class"]).to_csv(INPUT/"mutations.csv",index=False)
    px = rng.normal(5,0.8,(120,24)); px[:,12:] += (effect*0.6)[:,None]
    pp = ttest_ind(px[:,12:],px[:,:12],axis=1,equal_var=False).pvalue
    pd.DataFrame({"gene":genes,"log2FC":px[:,12:].mean(1)-px[:,:12].mean(1),"padj":bh(pp)}).to_csv(INPUT/"proteomics_de.csv",index=False)
    toy = nx.barabasi_albert_graph(120,3,seed=SEED)
    edges = [(genes[a],genes[b],round(float(rng.uniform(0.7,1)),4)) for a,b in toy.edges()]
    pd.DataFrame(edges,columns=["source","target","score"]).to_csv(INPUT/"ppi_edges.csv",index=False)
    drug_rows=[]; sig_rows=[]
    for d in range(8):
        did=f"DEMO_D{d:02d}"
        for g in rng.choice(genes,size=3,replace=False):
            drug_rows.append((did,f"Synthetic compound {d}",g,"unknown","SYNTHETIC","not biological evidence"))
        z = rng.normal(0,1,120) + (-effect if d<3 else effect*0.2)
        for g,v in zip(genes,z):
            sig_rows.append((f"DEMO_SIG{d:02d}",did,g,v,"SYNTHETIC_CELL","toy-dose","toy-time"))
    pd.DataFrame(drug_rows,columns=["drug_id","drug_name","gene","action","source","reference"]).to_csv(INPUT/"drug_targets.csv",index=False)
    pd.DataFrame(sig_rows,columns=["signature_id","drug_id","gene","z","cell_line","dose","time"]).to_csv(INPUT/"signatures.csv",index=False)
    (INPUT/"pathways.gmt").write_text("\n".join("\t".join([f"SYNTHETIC_SET_{i}","toy set"]+genes[i*15:(i+1)*15]) for i in range(8))+"\n")
    names=["expression_de.csv","samples.csv","mutations.csv","proteomics_de.csv","ppi_edges.csv","drug_targets.csv","signatures.csv","pathways.gmt"]
    pd.DataFrame([{"file":n,"source":"SYNTHETIC","release":"seed42","cohort":"24 toy samples","notes":"Generated only for software demonstration"} for n in names]).to_csv(INPUT/"provenance.csv",index=False)
    print("SYNTHETIC inputs generated in",INPUT)


## 4. Optional live resource helpers (real data preparation)
These helpers are **not called automatically**. Requests fail visibly and raw responses are saved. They do not silently substitute demo data.

STRING functional edges are associations, not necessarily direct physical binding; choose `network_type="physical"` if that is the scientific question. The helper retrieves an induced network only. Use a broader background interactome for proximity calculations, because a disease-only network biases distances. Full human networks can exceed Colab memory; document a justified background restriction and report disconnected targets.

TCGA discovery does not identify TNBC on its own and does not download or analyze raw counts. cBioPortal sample IDs and mutation profile must be selected from the same study; the helper returns raw records for normalization to the input contract. Open Targets evidence and DGIdb interactions are supplied by versioned exports under the above contract. Retain original references and action labels and do not equate an interaction with approval or efficacy.


In [ ]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent":"multiomics-colab-research/1.0"})

def request_json(method,url,**kwargs):
    r=SESSION.request(method,url,timeout=120,**kwargs)
    r.raise_for_status()
    data=r.json()
    if isinstance(data,dict) and data.get("errors"):
        raise RuntimeError(data["errors"])
    return data

def discover_tcga_brca():
    filters={"op":"and","content":[
        {"op":"in","content":{"field":"cases.project.project_id","value":["TCGA-BRCA"]}},
        {"op":"in","content":{"field":"data_type","value":["Gene Expression Quantification"]}}]}
    data=request_json("GET","https://api.gdc.cancer.gov/files",params={
        "filters":json.dumps(filters),"format":"JSON","size":20,
        "fields":"file_id,file_name,data_type,analysis.workflow_type"})
    (OUT/"gdc_discovery_raw.json").write_text(json.dumps(data,indent=2))
    print("Discovery preview only; pagination:",data.get("data",{}).get("pagination"))
    return pd.DataFrame(data["data"]["hits"])

def cbio_profiles(study_id):
    return pd.DataFrame(request_json("GET",f"https://www.cbioportal.org/api/studies/{study_id}/molecular-profiles"))

def cbio_mutations(molecular_profile_id,sample_ids):
    # Explicit sample list controls the cohort; retain all profiled IDs in samples.csv.
    rows=[]
    for start in range(0,len(sample_ids),100):
        batch=request_json("POST",f"https://www.cbioportal.org/api/molecular-profiles/{molecular_profile_id}/mutations/fetch",
            params={"projection":"DETAILED"},json={"sampleIds":list(sample_ids[start:start+100])})
        rows.extend(batch)
        time.sleep(0.2)
    (OUT/"cbio_mutations_raw.json").write_text(json.dumps(rows,indent=2))
    return pd.json_normalize(rows)

def string_network(symbols,network_type="functional"):
    if not 2<=len(symbols)<=1000: raise ValueError("Supply 2–1000 genes; use bulk downloads for large backgrounds")
    base="https://string-db.org/api/json"
    mapping=request_json("POST",base+"/get_string_ids",data={"identifiers":"\r".join(symbols),
                "species":9606,"limit":1,"echo_query":1,"caller_identity":"multiomics_colab"})
    mp=pd.DataFrame(mapping)
    if mp.empty: raise ValueError("No STRING identifiers mapped")
    mp.to_csv(OUT/"string_id_mapping.csv",index=False)
    mapped_queries=set(mp["queryItem"])
    pd.Series(sorted(set(symbols)-mapped_queries),name="unmapped_gene").to_csv(OUT/"string_unmapped.csv",index=False)
    if mp["stringId"].duplicated().any(): raise ValueError("Multiple inputs map to one STRING protein; resolve aliases")
    ids=mp["stringId"].tolist()
    data=request_json("POST",base+"/network",data={"identifiers":"\r".join(ids),"species":9606,
        "required_score":int(STRING_SCORE*1000),"network_type":network_type,
        "add_nodes":0,"caller_identity":"multiomics_colab"})
    (OUT/"string_raw.json").write_text(json.dumps(data,indent=2))
    if not data: raise ValueError("STRING returned no edges at the chosen threshold")
    df=pd.DataFrame(data)
    lookup=dict(zip(mp.stringId,mp.queryItem))
    result=pd.DataFrame({"source":df.stringId_A.map(lookup),"target":df.stringId_B.map(lookup),"score":df.score})
    if result[["source","target"]].isna().any().any(): raise ValueError("Unmapped returned STRING edge")
    return result

# Examples to run manually in real mode:
# display(discover_tcga_brca())
# display(cbio_profiles("brca_tcga_pan_can_atlas_2018"))
# raw = cbio_mutations(YOUR_PROFILE_ID, YOUR_PROFILED_TNBC_SAMPLE_IDS)
# edges = string_network(YOUR_BACKGROUND_HGNC_SYMBOLS)
# edges.to_csv(INPUT/"ppi_edges.csv",index=False)


## 5. Validate inputs and integrate molecular evidence
Missing optional proteomic evidence stays missing. RNA evidence is `abs(log2FC) × -log10(padj)`; mutation evidence is case mutation frequency. Percentile ranks give an equal-weight **heuristic** molecular score across observed layers. This score is not a p-value. Open Targets annotations are kept separate to avoid recycling prior knowledge into the molecular score.


In [ ]:
de=read_table("expression_de.csv",["gene","log2FC","pvalue","padj"])
samples=read_table("samples.csv",["sample_id","group","mutation_profiled"])
mut=read_table("mutations.csv",["sample_id","gene","variant_class"])
ppi=read_table("ppi_edges.csv",["source","target","score"])
dt=read_table("drug_targets.csv",["drug_id","drug_name","gene","action","source","reference"])
sig=read_table("signatures.csv",["signature_id","drug_id","gene","z","cell_line","dose","time"])
prot=read_table("proteomics_de.csv",["gene","log2FC","padj"],True)
ot=read_table("open_targets.csv",["gene","disease_id","association_score","source"],True)
prov=read_table("provenance.csv",["file","source","release","cohort","notes"])
for frame,cols,name in [(de,["gene"],"expression"),(samples,["sample_id"],"samples"),
                        (prot,["gene"],"proteomics"),(sig,["signature_id","gene"],"signatures")]:
    unique(frame,cols,name)
if dt.empty or sig.empty or de.empty: raise ValueError("Expression, drug-target and signature inputs must be nonempty")
for frame,cols,name in [(de,["log2FC","pvalue","padj"],"expression"),(prot,["log2FC","padj"],"proteomics"),
                        (ppi,["score"],"network"),(sig,["z"],"signatures")]:
    numeric(frame,cols,name)
for frame,cols in [(de,["pvalue","padj"]),(prot,["padj"]),(ppi,["score"])]:
    for col in cols:
        if not frame[col].between(0,1).all(): raise ValueError(f"{col} must be between 0 and 1")
for frame,cols in [(de,["gene"]),(mut,["sample_id","gene"]),(ppi,["source","target"]),
                    (dt,["drug_id","drug_name","gene","source","reference"]),(sig,["signature_id","drug_id","gene"])]:
    for col in cols:
        if frame[col].isna().any() or (frame[col].astype(str).str.strip()=="").any():
            raise ValueError(f"Missing identifier/evidence: {col}")
if not samples.group.isin(["case","control"]).all(): raise ValueError("Use case/control groups")
if not samples.mutation_profiled.isin([0,1]).all(): raise ValueError("mutation_profiled must be 0/1")
if not mut.variant_class.isin(["nonsynonymous","synonymous","unknown"]).all(): raise ValueError("Normalize variant_class first")
if set(mut.sample_id)-set(samples.sample_id): raise ValueError("Mutation sample missing from samples.csv")
if sig.groupby("signature_id")[["drug_id","cell_line","dose","time"]].nunique(dropna=False).gt(1).any().any():
    raise ValueError("Each signature must have exactly one compound and experimental context")
if dt.groupby("drug_id").drug_name.nunique().gt(1).any(): raise ValueError("Conflicting drug names for one ID")
if MODE=="real":
    for frame,col in [(de,"gene"),(ppi,"source"),(dt,"drug_id")]:
        if frame[col].astype(str).str.startswith("DEMO_").any(): raise ValueError("Synthetic inputs found in real mode")
    needed={"expression_de.csv","samples.csv","mutations.csv","ppi_edges.csv","drug_targets.csv","signatures.csv"}
    if not needed<=set(prov.file): raise ValueError("Record provenance for every required input")
profiled=set(samples.loc[(samples.group=="case") & (samples.mutation_profiled==1),"sample_id"])
if not profiled: raise ValueError("No mutation-profiled cases; supply the denominator")
case_calls=mut[mut.sample_id.isin(profiled) & (mut.variant_class=="nonsynonymous")]
mf=case_calls.drop_duplicates(["sample_id","gene"]).groupby("gene").size()/len(profiled)
evidence=de.set_index("gene").copy()
evidence["mutation_frequency"]=mf.reindex(evidence.index,fill_value=0)
evidence["rna_evidence"]=evidence.log2FC.abs()*(-np.log10(evidence.padj.clip(lower=1e-300)))
evidence["rna_rank"]=rank01(evidence.rna_evidence)
evidence["mutation_rank"]=rank01(evidence.mutation_frequency)
layers=["rna_rank","mutation_rank"]
if len(prot):
    protein=prot.set_index("gene")
    evidence["protein_log2FC"]=protein.log2FC
    evidence["protein_padj"]=protein.padj
    evidence["protein_rank"]=rank01(evidence.protein_log2FC.abs()*(-np.log10(evidence.protein_padj.clip(lower=1e-300))))
    evidence["rna_protein_direction_agrees"]=np.where(evidence.protein_log2FC.notna(),np.sign(evidence.log2FC)==np.sign(evidence.protein_log2FC),np.nan)
    layers.append("protein_rank")
evidence["observed_layers"]=evidence[layers].notna().sum(axis=1)
evidence["molecular_score"]=evidence[layers].mean(axis=1)
evidence["is_deg"]=(evidence.padj<FDR)&(evidence.log2FC.abs()>=MIN_ABS_LOG2FC)
if not evidence.is_deg.any(): raise ValueError("No DEGs at prespecified thresholds; inspect the data rather than silently relaxing criteria")
if len(ot):
    unique(ot,["gene"],"Open Targets"); numeric(ot,["association_score"],"Open Targets")
    if ot.disease_id.nunique()!=1 or not ot.association_score.between(0,1).all(): raise ValueError("Use one disease and scores in [0,1]")
    evidence=evidence.join(ot.set_index("gene").add_prefix("ot_"))
print(f"Tested genes: {len(de)}; DEGs: {evidence.is_deg.sum()}; profiled cases: {len(profiled)}")
display(evidence.sort_values("molecular_score",ascending=False).head(10))


## 6. Network modules, centrality and disease gene prioritization
All thresholded background edges are retained. Centrality is calculated on that graph, not solely on DEGs. Louvain modules use confidence weights; shortest paths later use unweighted hops (confidence is not a physical distance). Module overrepresentation uses tested genes mapped to the graph as its universe. Adjusted module p-values are exploratory because genes and network structure are not independent.


In [ ]:
ppi=ppi[(ppi.score>=STRING_SCORE)&(ppi.source!=ppi.target)].copy()
if ppi.empty: raise ValueError("No edges after filtering")
ppi["a"]=ppi[["source","target"]].min(axis=1)
ppi["b"]=ppi[["source","target"]].max(axis=1)
ppi=ppi.groupby(["a","b"],as_index=False).score.max().rename(columns={"a":"source","b":"target"})
G=nx.from_pandas_edgelist(ppi,"source","target",edge_attr="score")
communities=nx.community.louvain_communities(G,weight="score",seed=SEED)
module={g:i for i,c in enumerate(communities) for g in c}
centrality=pd.DataFrame({"degree":dict(G.degree()),"pagerank":nx.pagerank(G,weight="score"),"module":module})
evidence=evidence.join(centrality)
evidence["centrality_rank"]=rank01(evidence.pagerank)
evidence["integrated_gene_score"]=0.8*evidence.molecular_score+0.2*evidence.centrality_rank
universe=set(evidence.index)&set(G)
disease=set(evidence.index[evidence.is_deg])&set(G)
if not disease: raise ValueError("No disease genes mapped to network")
rows=[]
for i,c in enumerate(communities):
    measured=set(c)&universe; hits=measured&disease
    if measured:
        rows.append({"module":i,"network_size":len(c),"tested_size":len(measured),"deg_hits":len(hits),
                     "pvalue":hypergeom.sf(len(hits)-1,len(universe),len(disease),len(measured))})
modules=pd.DataFrame(rows); modules["padj"]=bh(modules.pvalue)
modules.sort_values("padj").to_csv(OUT/"modules.csv",index=False)
evidence.sort_values("integrated_gene_score",ascending=False).to_csv(OUT/"gene_priorities.csv")
pd.Series(sorted(set(evidence.index)-set(G)),name="unmapped_gene").to_csv(OUT/"network_unmapped_genes.csv",index=False)
print(f"Network: {len(G)} nodes / {G.number_of_edges()} edges / {nx.number_connected_components(G)} components")
print(f"Disease genes mapped: {len(disease)}/{int(evidence.is_deg.sum())}")
display(modules.sort_values("padj").head())
fig,ax=plt.subplots(figsize=(8,5))
ax.scatter(evidence.log2FC,-np.log10(evidence.padj.clip(lower=1e-300)),c=evidence.is_deg.map({True:"crimson",False:"gray"}),s=14)
ax.set(xlabel="Disease − control log2 fold change",ylabel="−log10 adjusted p",title=f"{MODE.upper()}: expression evidence")
fig.tight_layout(); fig.savefig(OUT/"volcano.png",dpi=160); plt.show()


## 7. Pathway enrichment: GO, Reactome, KEGG, Enrichr and g:Profiler
Use all tested expression genes as the enrichment universe. The local hypergeometric implementation intersects each gene set with that universe and applies BH across **all tested sets**, including zero-overlap sets. Live Enrichr library names are discovered rather than hard-coded to a possibly obsolete release. The library selected and gene sets are saved.

Demo mode uses toy pathways only. Real mode can use your versioned GMT or enable live enrichment. If neither is supplied, enrichment is explicitly skipped; other analyses continue.


In [ ]:
def read_gmt(path):
    result={}
    for line in path.read_text().splitlines():
        parts=line.split("\t")
        if len(parts)>=3: result[parts[0]]=set(parts[2:])
    return result

def ora(query,background,sets):
    bg=set(background); q=set(query)&bg; rows=[]
    for term,geneset in sets.items():
        s=set(geneset)&bg
        if len(s)<3 or len(s)>1000: continue
        overlap=q&s
        rows.append({"term":term,"overlap":len(overlap),"set_size":len(s),"query_size":len(q),
                     "universe_size":len(bg),"genes":";".join(sorted(overlap)),
                     "pvalue":float(hypergeom.sf(len(overlap)-1,len(bg),len(s),len(q)))})
    if not rows: return pd.DataFrame(columns=["term","overlap","pvalue","padj"])
    r=pd.DataFrame(rows); r["padj"]=bh(r.pvalue)
    return r.sort_values("padj")

sets=read_gmt(INPUT/"pathways.gmt") if (INPUT/"pathways.gmt").exists() else {}
if MODE=="real" and LIVE_ENRICHMENT:
    import gseapy as gp
    available=gp.get_library_name(organism="Human")
    selected=[]
    for prefix in ["GO_Biological_Process_","Reactome_","KEGG_"]:
        options=[v for v in available if v.startswith(prefix) and (prefix!="KEGG_" or "Human" in v)]
        if not options: raise ValueError(f"No library for {prefix}; inspect available names")
        selected.append(sorted(options)[-1])
    (OUT/"enrichr_library_names.json").write_text(json.dumps(selected,indent=2))
    for library in selected:
        library_sets=gp.get_library(name=library,organism="Human")
        sets.update({library+"::"+k:set(v) for k,v in library_sets.items()})
    from gprofiler import GProfiler
    gprof=GProfiler(return_dataframe=True)
    gp_result=gprof.profile(organism="hsapiens",query=evidence.index[evidence.is_deg].tolist(),
                          background=evidence.index.tolist(),domain_scope="custom",
                          sources=["GO:BP","REAC","KEGG"])
    gp_result.to_csv(OUT/"gprofiler_enrichment.csv",index=False)
if sets:
    (OUT/"used_pathways.gmt").write_text("\n".join("\t".join([k,"recorded gene set"]+sorted(v)) for k,v in sets.items()))
    enrichment=ora(evidence.index[evidence.is_deg],evidence.index,sets)
    enrichment.to_csv(OUT/"pathway_enrichment.csv",index=False)
    display(enrichment.head(15))
else:
    print("Enrichment skipped: supply pathways.gmt or enable LIVE_ENRICHMENT in real mode.")


## 8. Drug proximity with a degree-matched null model
For each compound, compute the average distance from each mapped target to its nearest disease gene. Smaller distance means closer. Disconnected targets are reported and the reachable-target fraction accompanies each score; comparisons with poor coverage are unreliable.

The null replaces the reachable targets with distinct graph nodes in the **same connected components and log2 degree bins**, preserving the number in each stratum. Disease genes stay fixed. Random sets may overlap disease genes just as observed targets may. This conditional, approximate degree-matched null is not a full replication of every published proximity method. Degenerate nulls receive missing z-scores. Empirical one-sided p-values use a +1 correction and BH across drugs. With 200 draws, the minimum p-value is 1/201; increase permutations for research.


In [ ]:
component={g:i for i,c in enumerate(nx.connected_components(G)) for g in c}
dist=nx.multi_source_dijkstra_path_length(G,list(disease),weight=None)
# Pool includes every background node reachable from the disease set.
stratum={g:(component[g],int(np.floor(np.log2(max(1,G.degree(g)))))) for g in dist}
pools={}
for g,k in stratum.items(): pools.setdefault(k,[]).append(g)
proximity=[]
for did,group in dt.groupby("drug_id",sort=True):
    all_targets=set(group.gene); mapped=all_targets&set(G); reachable=mapped&set(dist)
    record={"drug_id":did,"drug_name":group.drug_name.iloc[0],"n_targets":len(all_targets),
            "n_mapped":len(mapped),"n_reachable":len(reachable),
            "target_coverage":len(reachable)/len(all_targets),
            "unmapped_targets":";".join(sorted(all_targets-set(G))),
            "disconnected_targets":";".join(sorted(mapped-set(dist)))}
    record.update(distance=np.nan,proximity_z=np.nan,proximity_p=np.nan)
    if reachable:
        observed=float(np.mean([dist[t] for t in reachable]))
        counts={}
        for t in reachable: counts[stratum[t]]=counts.get(stratum[t],0)+1
        null=[]
        for _ in range(N_PERM):
            random_targets=[]
            for k,n in counts.items(): random_targets.extend(rng.choice(pools[k],size=n,replace=False))
            null.append(np.mean([dist[t] for t in random_targets]))
        null=np.asarray(null); sd=null.std(ddof=1)
        record.update(distance=observed,proximity_z=(observed-null.mean())/sd if sd>0 else np.nan,
                      proximity_p=(1+np.sum(null<=observed))/(N_PERM+1),null_sd=sd)
    proximity.append(record)
proximity=pd.DataFrame(proximity)
proximity["proximity_q"]=bh(proximity.proximity_p)
proximity.to_csv(OUT/"drug_network_proximity.csv",index=False)
display(proximity.sort_values("proximity_z").head(10))


## 9. Expression signature reversal
Calculate Spearman correlation across all overlapping tested genes, not only significant genes. Negative correlation supports reversal. A gene-label permutation yields an exploratory one-sided p-value; gene correlation and experimental context limit its interpretation. Adjust across all signatures. Multiple signatures per drug are summarized by the median reversal; no drug-level significance is claimed. Missing or constant signatures stay unavailable, never become zero evidence.


In [ ]:
reversals=[]
for sid,group in sig.groupby("signature_id",sort=True):
    joined=evidence[["log2FC"]].join(group.set_index("gene")[["z"]],how="inner").dropna()
    rec={"signature_id":sid,"drug_id":group.drug_id.iloc[0],"cell_line":group.cell_line.iloc[0],
         "dose":group.dose.iloc[0],"time":group.time.iloc[0],"n_genes":len(joined),
         "rho":np.nan,"reversal":np.nan,"pvalue":np.nan,"status":"insufficient_overlap_or_constant"}
    if len(joined)>=MIN_SIGNATURE_GENES and joined.log2FC.nunique()>1 and joined.z.nunique()>1:
        # Rank once to make permutations much faster than repeated Spearman calls.
        a=joined.log2FC.rank().to_numpy(); b=joined.z.rank().to_numpy()
        a=(a-a.mean())/np.linalg.norm(a-a.mean()); b=(b-b.mean())/np.linalg.norm(b-b.mean())
        rho=float(a@b)
        null=np.array([a@rng.permutation(b) for _ in range(N_PERM)])
        rec.update(rho=rho,reversal=-rho,pvalue=(1+np.sum(null<=rho))/(N_PERM+1),status="ok")
    reversals.append(rec)
rev=pd.DataFrame(reversals)
if rev.empty: raise ValueError("No signatures supplied")
rev["padj"]=bh(rev.pvalue)
rev.to_csv(OUT/"signature_reversal.csv",index=False)
drug_rev=rev.groupby("drug_id").agg(reversal=("reversal","median"),n_signatures=("signature_id","size"),
                                  n_valid_signatures=("reversal","count"),min_overlap=("n_genes","min"))
display(rev.sort_values("reversal",ascending=False).head(10))


## 10. Combine evidence and inspect candidates
The primary score is an equal-weight mean of three percentile ranks: negative proximity z, median signature reversal, and mean molecular evidence among targets. This is a transparent heuristic, not a learned predictor or calibrated probability. Require all three channels plus at least 50% reachable-target coverage; incomplete compounds remain in the output with a reason and no rank. Sensitivity columns change one component's weight to 60% to reveal unstable ordering.

Check interaction mechanism, loss/gain of function, tissue exposure, dose plausibility, toxicity, approval status and literature independently. A highly mutated tumor suppressor is not automatically an inhibition target. Drug directionality is retained for review but not inferred by this score.


In [ ]:
target_scores=[]
for did,group in dt.groupby("drug_id"):
    vals=evidence.molecular_score.reindex(sorted(set(group.gene)))
    target_scores.append({"drug_id":did,"target_molecular_evidence":vals.mean(),
                          "n_targets_with_omics":int(vals.notna().sum()),
                          "actions":";".join(sorted(set(group.action.fillna("unknown").astype(str)))),
                          "evidence_sources":";".join(sorted(set(group.source.astype(str))))})
ranked=proximity.merge(drug_rev,on="drug_id",how="left").merge(pd.DataFrame(target_scores),on="drug_id",how="left")
ranked["network_rank"]=rank01(-ranked.proximity_z)
ranked["reversal_rank"]=rank01(ranked.reversal)
ranked["omics_rank"]=rank01(ranked.target_molecular_evidence)
channels=["network_rank","reversal_rank","omics_rank"]
eligible=ranked[channels].notna().all(axis=1)&(ranked.target_coverage>=0.5)
ranked["status"]=np.where(eligible,"ranked","insufficient_evidence_or_coverage")
ranked["combined_score"]=ranked[channels].mean(axis=1).where(eligible)
ranked["rank"]=ranked.combined_score.rank(ascending=False,method="min").astype("Int64")
for emphasis in channels:
    weights=np.array([0.6 if c==emphasis else 0.2 for c in channels])
    ranked[f"sensitivity_{emphasis}"]=(ranked[channels]*weights).sum(axis=1,min_count=3).where(eligible)
ranked["data_mode"]=MODE
ranked=ranked.sort_values("combined_score",ascending=False,na_position="last")
ranked.to_csv(OUT/"ranked_drug_candidates.csv",index=False)
display(ranked[["rank","drug_name","combined_score","proximity_z","proximity_q","reversal","target_coverage","status"]])
valid=ranked.dropna(subset=["combined_score"]).head(10)
if len(valid):
    fig,ax=plt.subplots(figsize=(9,5))
    ax.barh(valid.drug_name.iloc[::-1],valid.combined_score.iloc[::-1],color="#177f89")
    ax.set(xlabel="Heuristic combined score",title=f"{MODE.upper()}: research candidates")
    fig.tight_layout(); fig.savefig(OUT/"candidate_ranking.png",dpi=160); plt.show()


## 11. Cytoscape exports and reproducibility
Open Cytoscape on your computer → **File → Import → Network from File → `network.graphml`**. For a separate table, import `network_nodes.csv` keyed by `gene`. Use `module` for color, `pagerank` for size and `log2FC` for expression overlay. Colab cannot directly control a Cytoscape desktop running on your laptop without an explicitly configured connection.

The ZIP contains results, input snapshots, parameter settings, hashes and package versions. It contains no credentials. Download it before the Colab runtime expires.


In [ ]:
node_table=centrality.join(evidence.drop(columns=["degree","pagerank","module"]),how="left")
node_table.index.name="gene"
node_table.to_csv(OUT/"network_nodes.csv")
ppi.to_csv(OUT/"network_edges.csv",index=False)
dt.to_csv(OUT/"drug_target_evidence.csv",index=False)
for g,row in node_table.iterrows():
    for key,value in row.items():
        if pd.notna(value):
            if isinstance(value,np.generic): value=value.item()
            G.nodes[g][key]=value if isinstance(value,(str,bool,int,float)) else str(value)
nx.write_graphml(G,OUT/"network.graphml")
# Small disease-focused visualization; full graph is retained in GraphML.
plot_nodes=sorted(disease,key=lambda g:G.degree(g),reverse=True)[:60]
H=G.subgraph(plot_nodes)
fig,ax=plt.subplots(figsize=(10,8))
pos=nx.spring_layout(H,seed=SEED,weight="score")
nx.draw_networkx(H,pos,ax=ax,node_color=[module[g] for g in H],cmap="tab20",node_size=160,
                 font_size=6,width=0.5,edge_color="#bbbbbb")
ax.set_title(f"{MODE.upper()}: disease-gene subgraph (display subset)"); ax.axis("off")
fig.tight_layout(); fig.savefig(OUT/"disease_network.png",dpi=160); plt.show()
input_snapshot=OUT/"inputs"; input_snapshot.mkdir(exist_ok=True)
hashes={}
for path in INPUT.iterdir():
    if path.is_file():
        hashes[path.name]=hashlib.sha256(path.read_bytes()).hexdigest()
        shutil.copy2(path,input_snapshot/path.name)
packages=["numpy","pandas","scipy","networkx","matplotlib","requests","gseapy","gprofiler-official"]
manifest={"timestamp_utc":datetime.now(timezone.utc).isoformat(),"mode":MODE,"seed":SEED,"permutations":N_PERM,
          "fdr":FDR,"min_abs_log2fc":MIN_ABS_LOG2FC,"string_score":STRING_SCORE,
          "min_signature_genes":MIN_SIGNATURE_GENES,"live_enrichment":LIVE_ENRICHMENT,
          "python":platform.python_version(),"packages":{p:importlib.metadata.version(p) for p in packages},
          "input_sha256":hashes,"proximity_null":"fixed disease; component/log2-degree-stratified target resampling",
          "ranking_weights":{"network":1/3,"reversal":1/3,"target_omics":1/3},
          "mutation_profiled_cases":len(profiled),"network_nodes":len(G),"disease_mapped":len(disease)}
(OUT/"run_manifest.json").write_text(json.dumps(manifest,indent=2))
(OUT/"requirements-resolved.txt").write_text("\n".join(f"{p}=={manifest['packages'][p]}" for p in packages)+"\n")
archive=shutil.make_archive(str(OUT),"zip",root_dir=OUT)
print("Results:",OUT.resolve()); print("ZIP:",archive)
# Run manually to download in Colab:
# from google.colab import files
# files.download(archive)


## 12. References, limitations and next research steps

- [STRING API documentation](https://string-db.org/help/api/) — identifier mapping, network types and versioned endpoints.
- [GDC API documentation](https://docs.gdc.cancer.gov/API/Users_Guide/Getting_Started/) — TCGA file discovery and access.
- [cBioPortal API](https://docs.cbioportal.org/web-api-and-clients/) — profile and mutation data access.
- [GSEApy documentation](https://gseapy.readthedocs.io/en/latest/) and [g:Profiler APIs](https://biit.cs.ut.ee/gprofiler/page/apis) — gene-set retrieval and enrichment.
- [DGIdb Python client](https://github.com/dgidb/dgipy) — current interaction access documentation; this notebook accepts curated exports.
- [Open Targets access](https://platform-docs.opentargets.org/data-access) — versioned evidence exports.
- [LINCS signature data levels](https://clue.io/connectopedia/data_levels) — level-5 perturbation signatures. [CLUE](https://clue.io/) announces retirement of its tools effective January 31, 2026; this notebook therefore uses downloaded signature subsets and does not depend on a live CLUE query endpoint. Verify availability and licensing of any archival release used.
- [Cytoscape](https://cytoscape.org/) and [NetworkX](https://networkx.org/documentation/stable/) — network analysis and visualization.

**Before a real study:** predefine the contrast, cohort, thresholds and drug universe; resolve gene identifiers; document mutation assay coverage; select LINCS context without looking at the result; record all data releases. Evaluate sensitivity to network source, edge threshold, null matching, signature context and score weights. Validate candidates against independent perturbation experiments and external cohorts. No classifier, cross-validation performance, clinical validation or docking result is claimed by this notebook.

**Troubleshooting:** a missing-column error means the uploaded schema differs from the table above. A zero mapped-gene result usually indicates incompatible identifiers. Empty signatures or constant vectors cannot support reversal. An API failure is not a negative biological result; retry later or import a versioned local export. Real analysis can be slow with large networks or many compounds: start with a predefined pilot subset and scale deliberately.
